In [1]:
!nvidia-smi

Thu Jul  9 19:11:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers
!pip install -q datasets
!pip install -q peft
!pip install -q trl
!pip install -q accelerate
!pip install -q bitsandbytes
!pip install -q sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00


In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "Anupam007/indian-recipe-dataset",
    split="train"
)

Cleaned_Indian_Food_Dataset.csv:   0%|          | 0.00/11.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5938 [00:00<?, ? examples/s]

In [4]:
print(dataset)

Dataset({
    features: ['TranslatedRecipeName', 'TranslatedIngredients', 'TotalTimeInMins', 'Cuisine', 'TranslatedInstructions', 'URL', 'Cleaned-Ingredients', 'image-url', 'Ingredient-count'],
    num_rows: 5938
})


In [5]:
print(dataset.features)

{'TranslatedRecipeName': Value('string'), 'TranslatedIngredients': Value('string'), 'TotalTimeInMins': Value('int64'), 'Cuisine': Value('string'), 'TranslatedInstructions': Value('string'), 'URL': Value('string'), 'Cleaned-Ingredients': Value('string'), 'image-url': Value('string'), 'Ingredient-count': Value('int64')}


In [6]:
dataset[0]

{'TranslatedRecipeName': 'Masala Karela Recipe',
 'TranslatedIngredients': '1 tablespoon Red Chilli powder,3 tablespoon Gram flour (besan),2 teaspoons Cumin seeds (Jeera),1 tablespoon Coriander Powder (Dhania),2 teaspoons Turmeric powder (Haldi),Salt - to taste,1 tablespoon Amchur (Dry Mango Powder),6 Karela (Bitter Gourd/ Pavakkai) - deseeded,Sunflower Oil - as required,1 Onion - thinly sliced',
 'TotalTimeInMins': 45,
 'Cuisine': 'Indian',
 'TranslatedInstructions': 'To begin making the Masala Karela Recipe,de-seed the karela and slice.\nDo not remove the skin as the skin has all the nutrients.\nAdd the karela to the pressure cooker with 3 tablespoon of water, salt and turmeric powder and pressure cook for three whistles.\nRelease the pressure immediately and open the lids.\nKeep aside.Heat oil in a heavy bottomed pan or a kadhai.\nAdd cumin seeds and let it sizzle.Once the cumin seeds have sizzled, add onions and saute them till it turns golden brown in color.Add the karela, red chi

teaching the llm to how to take data as it cant take row and columns (user and assistent)

In [7]:
recipe = dataset[0]

for key, value in recipe.items():
    print("=" * 50)
    print(key)#key value pairs like dictionary in python
    print(value)

TranslatedRecipeName
Masala Karela Recipe
TranslatedIngredients
1 tablespoon Red Chilli powder,3 tablespoon Gram flour (besan),2 teaspoons Cumin seeds (Jeera),1 tablespoon Coriander Powder (Dhania),2 teaspoons Turmeric powder (Haldi),Salt - to taste,1 tablespoon Amchur (Dry Mango Powder),6 Karela (Bitter Gourd/ Pavakkai) - deseeded,Sunflower Oil - as required,1 Onion - thinly sliced
TotalTimeInMins
45
Cuisine
Indian
TranslatedInstructions
To begin making the Masala Karela Recipe,de-seed the karela and slice.
Do not remove the skin as the skin has all the nutrients.
Add the karela to the pressure cooker with 3 tablespoon of water, salt and turmeric powder and pressure cook for three whistles.
Release the pressure immediately and open the lids.
Keep aside.Heat oil in a heavy bottomed pan or a kadhai.
Add cumin seeds and let it sizzle.Once the cumin seeds have sizzled, add onions and saute them till it turns golden brown in color.Add the karela, red chilli powder, amchur powder, coriander

take the required ingredients

In [8]:
def create_prompt(example):

    prompt = f"""
User:
How do I make {example['TranslatedRecipeName']}?

Assistant:
Ingredients:
{example['TranslatedIngredients']}

Instructions:
{example['TranslatedInstructions']}
"""

    return {"text": prompt}

If the dataset has

6000 recipes

this function will be called

6000 times.

so we are applying to entire dataset.

In [9]:
dataset = dataset.map(create_prompt)

Map:   0%|          | 0/5938 [00:00<?, ? examples/s]

In [10]:
print(dataset[15]["text"])


User:
How do I make South Indian Onion Chutney Recipe - South Indian Onion Chutney (Recipe)?

Assistant:
Ingredients:
 1 teaspoon cumin seeds, 2 teaspoons oil, 2 tablespoons black urad dal (split), salt - 1 teaspoon, 3 dry red chillies, 1/2 teaspoon oil, 1 tablespoon tamarind paste, as per taste Rye, 1 sprig curry leaves, 1/2 teaspoon jaggery,2 onions

Instructions:
To make South Indian Onion Chutney, first of all chop the onion and keep it aside.
Now heat 1 teaspoon of oil in the pan.
Add cumin seeds, dry red chillies and let it cook for 10 seconds.
Now add urad dal in it and let it cook till it becomes golden.
Turn off the gas and drain in a bowl.
Add another spoon of oil to the same pan.
Add onions and let it cook for 4 to 5 minutes.
Turn off the gas and let it cool down.
Now put urad dal, cumin and dry red chillies in the mixer grinder and grind them.
Add cooked onions, tamarind and jaggery.
Add some water and grind it.
Now for the tempering, heat the oil in a small pan.
Add musta

loading the model

In [11]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

tokenizer

In [12]:
text = "How do I make dosa?"

tokens = tokenizer(text)

print(tokens)

{'input_ids': [4340, 653, 358, 1281, 8750, 64, 30], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}


In [13]:
decoded = tokenizer.decode(tokens["input_ids"])

print(decoded)

How do I make dosa?


pre training a step just before fine tune

In [14]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

we can compair with the results

In [15]:
messages = [
    {
        "role": "user",
        "content": "How do I make Paneer Butter Masala?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
How do I make Paneer Butter Masala?
assistant
Paneer Butter Masala is a popular Indian dish that combines tender paneer (Indian cheese) with rich and flavorful spices and butter. Here’s a simple recipe to make Paneer Butter Masala:

### Ingredients:
- 1 cup paneer (Indian cheese)
- 2 tablespoons butter or ghee
- 2 tablespoons oil
- 1 onion, finely chopped
- 3-4 cloves garlic, minced
- 2-3 green chilies, slit and finely chopped (optional)
- 1 teaspoon cumin seeds
- 1 teaspoon turmeric powder
- 1 teaspoon coriander powder
- 1/2 teaspoon red chili powder (adjust according to spice preference)
- Salt to taste
- 1 teaspoon garam masala
- 1 teaspoon chaat masala
- 1/2 teaspoon black pepper
- 1/2 teaspoon asafoetida (hing)
- 1 cup water
- Fresh cilantro


fine turning starts

In [16]:
def create_messages(example):
    return {
        "messages": [
            {
                "role": "user",
                "content": f"How do I make {example['TranslatedRecipeName']}?"
            },
            {
                "role": "assistant",
                "content":
                f"""Ingredients:
{example['TranslatedIngredients']}

Instructions:
{example['TranslatedInstructions']}"""
            }
        ]
    }

In [17]:
dataset = dataset.map(create_messages)

Map:   0%|          | 0/5938 [00:00<?, ? examples/s]

model always trains like this:
<|im_start|>user
Question
<|im_end|>

<|im_start|>assistant
Answer
<|im_end|>

In [18]:
dataset[22]["messages"]

[{'role': 'user',
  'content': 'How do I make Homemade Healthy Subway Sandwich Recipe With Beet & Sprout?'},
 {'role': 'assistant',
  'content': 'Ingredients:\n2 Tomatoes - thinly sliced,1 Beetroot - grated,1/2 cup Hung Curd (Greek Yogurt),2 Submarine Bread (Subway Bread) - (flat breads or foot longs),2 cloves Garlic - grated,Tabasco Original - Hot Sauce - to taste,Salt and Pepper - to taste,2 Stalks Spring Onion (Bulb & Greens) - finely chopped,1/2 cup Green Moong Sprouts,2 tablespoon Coriander (Dhania) Leaves - finely chopped\n\nInstructions:\nTo begin making Subway Sandwich Recipe With Roasted Beetroot, we will first cook the beets.Heat a teaspoon of oil in a wok; add the grated beets and garlic, sprinkle some salt and stir fry on low to medium heat until the beets are softened and cooked.Once you notice the beetroot is cooked through, turn off the heat and allow it to cool completely.Once the beetroot has cooled completely, add it to a large mixing bowl.\nTo this add in the greek y

In [19]:
text = tokenizer.apply_chat_template(
    dataset[22]["messages"],
    tokenize=False
)

print(text)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
How do I make Homemade Healthy Subway Sandwich Recipe With Beet & Sprout?<|im_end|>
<|im_start|>assistant
Ingredients:
2 Tomatoes - thinly sliced,1 Beetroot - grated,1/2 cup Hung Curd (Greek Yogurt),2 Submarine Bread (Subway Bread) - (flat breads or foot longs),2 cloves Garlic - grated,Tabasco Original - Hot Sauce - to taste,Salt and Pepper - to taste,2 Stalks Spring Onion (Bulb & Greens) - finely chopped,1/2 cup Green Moong Sprouts,2 tablespoon Coriander (Dhania) Leaves - finely chopped

Instructions:
To begin making Subway Sandwich Recipe With Roasted Beetroot, we will first cook the beets.Heat a teaspoon of oil in a wok; add the grated beets and garlic, sprinkle some salt and stir fry on low to medium heat until the beets are softened and cooked.Once you notice the beetroot is cooked through, turn off the heat and allow it to cool completely.Once the beetroot has cooled

LoRA(low rank adaptation)
in 3B paramater we would only train 5 to 10M parameters
that is less that 1% of the model

In [20]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

In [21]:
from peft import get_peft_model

model = get_peft_model(model, peft_config)

In [22]:
model.print_trainable_parameters()

trainable params: 7,372,800 || all params: 3,093,311,488 || trainable%: 0.2383


dataset after fine tuning


In [23]:
def formatting_func(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False
        )
    }

dataset = dataset.map(formatting_func)

Map:   0%|          | 0/5938 [00:00<?, ? examples/s]

In [24]:
print(dataset[14]["text"])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
How do I make Saunf Aloo (Fennel Potato Curry) Recipe?<|im_end|>
<|im_start|>assistant
Ingredients:
1/4 cup Fresh cream - or 1/2 cup milk,5 Potatoes (Aloo) - halved with skin,2 teaspoon Fennel seeds (Saunf) - crushed,4 sprig Coriander (Dhania) Leaves - finely chopped,1 teaspoon Turmeric powder (Haldi),2 teaspoon Red Chilli powder

Instructions:
To begin with Saunf Aloo, heat oil in a pressure cooker.
Add turmeric powder, salt, red chilli powder and crushed fennel seeds till the fennel seeds turns golden in colour.Now add the potatoes and mix it properly with the masala.
Sauce it for 2-3 minutes and then add enough water to cover the potatoes in the cooker.Switch the heat after two whistles and once the pressure is released, open the lid of the pressure cooker.The next step is to add the cream or milk and mash a few potatoes to get a thicker consistency.
Let it cook for 5 m

In [25]:
columns_to_keep = ["text"]

dataset = dataset.remove_columns(
    [col for col in dataset.column_names if col not in columns_to_keep]
)

In [26]:
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 5938
})


dataset training and testing

In [27]:
dataset = dataset.train_test_split(
    test_size=0.05,
    seed=42
)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['text'],
    num_rows: 5641
})
Dataset({
    features: ['text'],
    num_rows: 297
})


In [28]:
!pip install -q -U trl

In [29]:
from trl import SFTTrainer
from transformers import TrainingArguments

In [30]:
training_args = TrainingArguments(
    output_dir="./recipe_qwen",

    num_train_epochs=1,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=10,

    save_strategy="epoch",

    eval_strategy="epoch",

    fp16=True,

    report_to="none"
)

In [31]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    args=training_args
)

Adding EOS to train dataset:   0%|          | 0/5641 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5641 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5641 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5641 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/297 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/297 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/297 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/297 [00:00<?, ? examples/s]

In [33]:
print(len(train_dataset))
print(len(test_dataset))

5641
297


In [32]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


OutOfMemoryError: CUDA out of memory. Tried to allocate 44.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 29.81 MiB is free. Including non-PyTorch memory, this process has 14.53 GiB memory in use. Of the allocated memory 14.34 GiB is allocated by PyTorch, and 59.60 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)